<a href="https://colab.research.google.com/github/shanusushmita/FairSearch/blob/main/FairSearch_Metric_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

def calculate_mrr(relevance_scores):
    """Calculates Mean Reciprocal Rank for a single query."""
    for i, rel in enumerate(relevance_scores):
        if rel > 0: # Assuming binary relevance
            return 1 / (i + 1)
    return 0

def calculate_exposure(protected_attributes):
    """
    Calculates the average exposure (attention) for each group.
    Standard IR model: Exposure = 1 / log2(rank + 1).
    """
    exposures = {val: [] for val in np.unique(protected_attributes)}
    for i, attr in enumerate(protected_attributes):
        rank = i + 1
        exposure = 1 / np.log2(rank + 1)
        exposures[attr].append(exposure)

    return {group: np.mean(val) for group, val in exposures.items()}

def calculate_selection_rate(protected_attributes, k=10):
    """
    Calculates the proportion of each group in the top K results.
    Used for Demographic Parity analysis.
    """
    top_k_attrs = protected_attributes[:k]
    # Proportion of each group in the top K
    return pd.Series(top_k_attrs).value_counts(normalize=True).to_dict()

# --- EXAMPLE USAGE WITH SIMULATED DATA ---
# Assume 0 = Underrepresented Institution, 1 = Elite/Privileged
# A biased system might rank all '1's at the top.

results = pd.DataFrame({
    'doc_id': range(10),
    'is_privileged': [1, 1, 1, 0, 0, 0, 0, 0, 0, 0], # Top 3 are Elite
    'relevance': [1, 0, 1, 1, 0, 0, 1, 1, 0, 1]      # Actual ground truth relevance
})

# 1. Calculate Utility
y_true = np.array([results['relevance'].values])
y_score = np.array([np.arange(10, 0, -1)]) # Mock scores 10, 9, 8...
ndcg = ndcg_score(y_true, y_score)
mrr = calculate_mrr(results['relevance'].values)

# 2. Calculate Fairness
exposure = calculate_exposure(results['is_privileged'].values)
parity_at_5 = calculate_selection_rate(results['is_privileged'].values, k=5)

print(f"--- Utility Metrics ---")
print(f"NDCG@10: {ndcg:.4f}")
print(f"MRR:     {mrr:.4f}")

print(f"\n--- Fairness Metrics ---")
print(f"Average Exposure (Privileged vs Underrepresented): {exposure}")
print(f"Demographic Parity @ Top 5: {parity_at_5}")